# Predicting Stellar Class — LightGBM + Optuna

Feature pipeline from v4/v5. Hyperparameters tuned with Optuna (80k stratified sample, 3-fold CV, early stopping). Metric: **balanced accuracy**. Then train on full data and submit.

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score

import lightgbm as lgb
from lightgbm import LGBMClassifier

try:
    import optuna
    OPTUNA_OK = True
    print(f'Optuna {optuna.__version__}')
except ImportError:
    OPTUNA_OK = False
    print('pip install optuna')

print(f'LightGBM {lgb.__version__}')

## Configuration

In [ ]:
ID_COL = 'id'
TARGET_COL = 'class'
CATEGORICAL_COLS = ['spectral_type', 'galaxy_population']

DATA_DIR = '/kaggle/input/competitions/playground-series-s6e6'
if not os.path.isdir(DATA_DIR):
    DATA_DIR = 'data'

N_TRIALS = 30
TUNE_SAMPLE_SIZE = 80_000
NUM_BOOST_ROUND = 800
EARLY_STOPPING_ROUNDS = 50

## Load Data

Competition files at `/kaggle/input/competitions/playground-series-s6e6/`. Locally, fallback to `data/`.

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print(f'Train: {train_df.shape}, Test: {test_df.shape}')

## Feature Engineering

Derived features:
- Color indices (u-g, g-r, r-i, r-z)
- 3D unit-sphere coordinates and sin/cos transforms
- Coordinate distance and bins
- Absolute magnitude in r-band (`abs_mag_r`)
- Log-redshift (`redshift_log`) — raw redshift is dropped

In [ ]:
def get_absolute_magnitude(m_apparent, z):
    z = np.asarray(z, dtype=float)
    m_apparent = np.asarray(m_apparent, dtype=float)
    mu = np.zeros_like(z, dtype=float)
    mask = z > 1e-4
    if mask.any():
        z_valid = z[mask]
        d_L = (299792.458 / 70) * z_valid * (1 + 0.775 * z_valid)
        mu[mask] = 5 * np.log10(d_L * 1e6) - 5
    return m_apparent - mu

def add_features(df):
    d = df.copy()
    d['color_ug'] = d['u'] - d['g']
    d['color_gr'] = d['g'] - d['r']
    d['color_ri'] = d['r'] - d['i']
    d['color_rz'] = d['r'] - d['z']
    d['coord_dist'] = np.sqrt(d['alpha']**2 + d['delta']**2)
    d['sin_alpha'] = np.sin(np.radians(d['alpha']))
    d['cos_alpha'] = np.cos(np.radians(d['alpha']))
    d['sin_delta'] = np.sin(np.radians(d['delta']))
    d['cos_delta'] = np.cos(np.radians(d['delta']))
    d['coord_x'] = np.cos(np.radians(d['delta'])) * np.cos(np.radians(d['alpha']))
    d['coord_y'] = np.cos(np.radians(d['delta'])) * np.sin(np.radians(d['alpha']))
    d['coord_z'] = np.sin(np.radians(d['delta']))
    d['alpha_bin'] = pd.cut(d['alpha'], bins=10, labels=False)
    d['delta_bin'] = pd.cut(d['delta'], bins=10, labels=False)
    d['abs_mag_r'] = get_absolute_magnitude(d['r'], d['redshift'])
    d['redshift_log'] = np.log1p(d['redshift'])
    d = d.drop(columns=['redshift'])
    d['coord_x_redshift'] = d['coord_dist'] * d['redshift_log']
    return d

train_df = add_features(train_df)
test_df = add_features(test_df)

print(f'Train: {train_df.shape}, Test: {test_df.shape}')

## Target Encoding

Categorical columns are replaced with smoothed class probabilities per category (one column per target class).

In [ ]:
def target_encode_multi(train_df, test_df, col, target_col, smoothing=10):
    classes = sorted(train_df[target_col].unique())
    global_probs = train_df[target_col].value_counts(normalize=True)
    agg = train_df.groupby(col)[target_col].value_counts(normalize=True).unstack(fill_value=0)
    counts = train_df.groupby(col)[target_col].count()
    for cls in classes:
        if cls not in agg.columns:
            agg[cls] = 0.0
        agg[cls] = (agg[cls] * counts + smoothing * global_probs.get(cls, 0)) / (counts + smoothing)
        train_df[f'{col}_{cls}_prob'] = train_df[col].map(agg[cls])
        test_df[f'{col}_{cls}_prob'] = test_df[col].map(agg[cls])
    return train_df, test_df

train_df, test_df = target_encode_multi(train_df, test_df, 'spectral_type', TARGET_COL, smoothing=20)
train_df, test_df = target_encode_multi(train_df, test_df, 'galaxy_population', TARGET_COL, smoothing=20)
train_df = train_df.drop(columns=CATEGORICAL_COLS)
test_df = test_df.drop(columns=CATEGORICAL_COLS)

## Prepare Matrices

In [ ]:
X_train = train_df.drop(columns=[ID_COL, TARGET_COL])
y_train = train_df[TARGET_COL]
X_test = test_df.drop(columns=[ID_COL])
test_ids = test_df[ID_COL]

le = LabelEncoder()
y_enc = le.fit_transform(y_train)
X_arr = X_train.values.astype(np.float32)

## Optuna Hyperparameter Search

30 trials on 80k stratified sample, 3-fold CV, native LightGBM API with early stopping. `best_iteration` sets `n_estimators` for the final sklearn model.

In [ ]:
DEFAULT_LGB_PARAMS = {
    'num_leaves': 255,
    'learning_rate': 0.05,
    'max_depth': 12,
    'min_child_samples': 30,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'reg_alpha': 0.01,
    'reg_lambda': 0.01,
}

def stratified_sample(X, y, n, seed=42):
    rng = np.random.default_rng(seed)
    idx = []
    for cls in np.unique(y):
        ci = np.where(y == cls)[0]
        take = max(2, round(n * len(ci) / len(y)))
        idx.extend(rng.choice(ci, min(take, len(ci)), replace=False))
    idx = np.array(idx)
    rng.shuffle(idx)
    return X[idx], y[idx]

X_tune, y_tune = stratified_sample(X_arr, y_enc, TUNE_SAMPLE_SIZE)
CV3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

def lgb_objective(trial):
    params = {
        'num_leaves': trial.suggest_int('num_leaves', 31, 511, step=32),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 5, 18),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 1.0, log=True),
    }
    fold_scores = []
    for tr_i, va_i in CV3.split(X_tune, y_tune):
        dtrain = lgb.Dataset(X_tune[tr_i], label=y_tune[tr_i])
        dval = lgb.Dataset(X_tune[va_i], label=y_tune[va_i], reference=dtrain)
        lgb_p = {
            'objective': 'multiclass', 'num_class': 3, 'metric': 'multi_logloss',
            'verbose': -1, 'seed': 42, 'num_threads': -1, **params,
        }
        booster = lgb.train(
            lgb_p, dtrain, num_boost_round=NUM_BOOST_ROUND,
            valid_sets=[dval],
            callbacks=[
                lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False),
                lgb.log_evaluation(-1),
            ],
        )
        preds = booster.predict(X_tune[va_i]).argmax(axis=1)
        fold_scores.append(balanced_accuracy_score(y_tune[va_i], preds))
    return -np.mean(fold_scores)

if OPTUNA_OK:
    print(f'Optuna: {N_TRIALS} trials, sample={len(y_tune)}, 3-fold CV...')
    t0 = time.time()
    study = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    study.optimize(lgb_objective, n_trials=N_TRIALS, show_progress_bar=True)
    best_params = study.best_params
    print(f'Done in {time.time() - t0:.0f}s | Best BalAcc: {-study.best_value:.5f}')
else:
    best_params = DEFAULT_LGB_PARAMS.copy()
    print('Optuna not available — using defaults')

print('Best params:', best_params)

# best_iteration для n_estimators
tr_i, va_i = next(CV3.split(X_tune, y_tune))
dtrain = lgb.Dataset(X_tune[tr_i], label=y_tune[tr_i])
dval = lgb.Dataset(X_tune[va_i], label=y_tune[va_i], reference=dtrain)
lgb_p = {
    'objective': 'multiclass', 'num_class': 3, 'metric': 'multi_logloss',
    'verbose': -1, 'seed': 42, 'num_threads': -1, **best_params,
}
evals_result = {}
cb_final = lgb.train(
    lgb_p, dtrain, num_boost_round=NUM_BOOST_ROUND,
    valid_sets=[dval],
    callbacks=[
        lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False),
        lgb.log_evaluation(-1),
        lgb.record_evaluation(evals_result),
    ],
)
best_iteration = cb_final.best_iteration
print(f'Best iteration: {best_iteration}')

loss_vals = evals_result['valid_0']['multi_logloss']
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(loss_vals, color='steelblue', linewidth=1.5, label='Val Loss')
ax.axvline(best_iteration, color='red', linestyle='--', label=f'Best iter={best_iteration}')
ax.set_xlabel('Boosting Round')
ax.set_ylabel('Multi-class Log Loss')
ax.set_title('LightGBM Convergence (Early Stopping)')
ax.legend()
plt.tight_layout()
plt.show()

## Model Hyperparameters

In [ ]:
FINAL_PARAMS = {
    'n_estimators': best_iteration,
    'learning_rate': best_params['learning_rate'],
    'max_depth': best_params['max_depth'],
    'num_leaves': best_params['num_leaves'],
    'min_child_samples': best_params['min_child_samples'],
    'subsample': best_params['bagging_fraction'],
    'subsample_freq': best_params['bagging_freq'],
    'colsample_bytree': best_params['feature_fraction'],
    'reg_alpha': best_params['reg_alpha'],
    'reg_lambda': best_params['reg_lambda'],
    'min_gain_to_split': 0.01,
    'class_weight': 'balanced',
    'random_state': 42,
    'verbose': -1,
}

FINAL_PARAMS

## Train Model

Train LightGBM on the full training set with tuned hyperparameters.

In [ ]:
model = LGBMClassifier(**FINAL_PARAMS)
model.fit(
    X_train, y_train,
    callbacks=[lgb.log_evaluation(period=200)],
)

predictions = model.predict(X_test)
print(f'Predictions: {len(predictions)}')
print(pd.Series(predictions).value_counts())

## Submission

In [ ]:
submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET_COL: predictions,
})

out_path = 'submission_v4_optuna.csv'
if os.path.isdir('/kaggle/working'):
    out_path = '/kaggle/working/submission.csv'
submission.to_csv(out_path, index=False)
submission.head()